In [23]:
def min_delta_boundaries_with_ratio(nums, max_k):
    if not nums:
        return {k: {"boundaries": [], "cost": 0, "ratio": 0} for k in range(max_k + 1)}
    nums.sort()
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]
    
    # DP[i][j] = (min_cost, total_boundary_sum, last_split, boundaries)
    dp = [[(float('inf'), 0, -1, []) for _ in range(max_k + 1)] for _ in range(n + 1)]
    dp[0][0] = (0, 0, -1, [])  # Base case
    
    for j in range(1, max_k + 1):
        for i in range(1, n + 1):
            for m in range(i):
                # Delta cost for nums[m..i-1] assigned to nums[i-1]
                delta = (i - m) * nums[i - 1] - (prefix[i] - prefix[m])
                total_delta = dp[m][j - 1][0] + delta
                # Sum of boundaries (each assigned boundary is nums[i-1])
                total_boundary = dp[m][j - 1][1] + (i - m) * nums[i - 1]
                if total_delta < dp[i][j][0]:
                    new_boundaries = dp[m][j - 1][3] + [nums[i - 1]]
                    dp[i][j] = (total_delta, total_boundary, m, new_boundaries)
    
    # Compute the ratio for each k
    results = {}
    for k in range(1, max_k + 1):
        total_delta = dp[n][k][0]
        total_boundary = dp[n][k][1]
        boundaries = dp[n][k][3]
        ratio = total_delta / total_boundary if total_boundary != 0 else 0
        results[k] = {
            "boundaries": boundaries,
            "cost": total_delta,
            "total_boundary_sum": total_boundary,
            "ratio": ratio
        }
    return results

# Example Usage
nums = [1, 3, 5, 7, 9]
max_k = 3
result = min_delta_boundaries_with_ratio(nums, max_k)
for k in result:
    print(f"k={k}: {result[k]}")

k=1: {'boundaries': [9], 'cost': 20, 'total_boundary_sum': 45, 'ratio': 0.4444444444444444}
k=2: {'boundaries': [3, 9], 'cost': 8, 'total_boundary_sum': 33, 'ratio': 0.24242424242424243}
k=3: {'boundaries': [1, 5, 9], 'cost': 4, 'total_boundary_sum': 29, 'ratio': 0.13793103448275862}


In [34]:
import pandas as pd

df = pd.read_csv('/mydata/hongshu/traces/meta2024_50m.csv')

column_array = df['object_size'].values.tolist()



In [32]:
def min_delta_boundaries_optimized(nums, max_k):
    if not nums:
        return {k: {"boundaries": [], "cost": 0, "ratio": 0} for k in range(max_k + 1)}
    nums.sort()
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]

    dp = [[(float('inf'), 0)] * (max_k + 1) for _ in range(n + 1)]
    dp[0][0] = (0, 0)
    split = [[0] * (max_k + 1) for _ in range(n + 1)]  # Track optimal splits

    for j in range(1, max_k + 1):
        for i in range(n, 0, -1):  # Fill DP table backwards
            low = split[i - 1][j] if (i > 1 and j > 1) else 0
            high = split[i][j - 1] if (j > 1) else i - 1
            for m in range(low, high + 1):
                delta = (i - m) * nums[i - 1] - (prefix[i] - prefix[m])
                total_boundary = dp[m][j - 1][1] + (i - m) * nums[i - 1]
                if dp[m][j - 1][0] + delta < dp[i][j][0]:
                    dp[i][j] = (dp[m][j - 1][0] + delta, total_boundary)
                    split[i][j] = m

    # Reconstruct boundaries (if needed)
    results = {}
    for k in range(1, max_k + 1):
        total_delta = dp[n][k][0]
        total_boundary = dp[n][k][1]
        ratio = total_delta / total_boundary if total_boundary != 0 else 0
        results[k] = {"cost": total_delta, "ratio": ratio}
    return results

In [30]:
result = min_delta_boundaries_with_ratio([1, 3, 5, 7, 9], 3)
for k in result:
    print(f"k={k}: cost={result[k]['cost']}, ratio={result[k]['ratio']}")

k=1: cost=20, ratio=0.4444444444444444
k=2: cost=8, ratio=0.24242424242424243
k=3: cost=4, ratio=0.13793103448275862


In [35]:
result = min_delta_boundaries_with_ratio(column_array, 40)
for k in result:
    print(f"k={k}: cost={result[k]['cost']}, ratio={result[k]['ratio']}")

: 

In [31]:
def min_delta_boundaries_optimized(nums, max_k):
    if not nums:
        return {k: {"boundaries": [], "cost": 0, "ratio": 0} for k in range(max_k + 1)}
    nums.sort()
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]

    # DP[i][j] = (min_cost, total_boundary_sum, last_split)
    dp = [[(float('inf'), 0, -1)] * (max_k + 1) for _ in range(n + 1)]
    dp[0][0] = (0, 0, -1)
    split = [[0] * (max_k + 1) for _ in range(n + 1)]  # Track optimal splits

    for j in range(1, max_k + 1):
        for i in range(n, 0, -1):  # Fill DP table backwards
            low = split[i - 1][j] if (i > 1 and j > 1) else 0
            high = split[i][j - 1] if (j > 1) else i - 1
            for m in range(low, high + 1):
                delta = (i - m) * nums[i - 1] - (prefix[i] - prefix[m])
                total_boundary = dp[m][j - 1][1] + (i - m) * nums[i - 1]
                if dp[m][j - 1][0] + delta < dp[i][j][0]:
                    dp[i][j] = (dp[m][j - 1][0] + delta, total_boundary, m)
                    split[i][j] = m

    # Reconstruct boundaries for each k
    results = {}
    for k in range(1, max_k + 1):
        if dp[n][k][0] == float('inf'):
            results[k] = {"boundaries": [], "cost": float('inf'), "ratio": float('inf')}
            continue
        # Backtrack to find boundaries
        boundaries = []
        i, remaining_k = n, k
        while remaining_k > 0:
            m = dp[i][remaining_k][2]  # last_split for dp[i][remaining_k]
            boundaries.append(nums[i - 1])  # The boundary is nums[i-1]
            i = m
            remaining_k -= 1
        boundaries = sorted(boundaries)  # Optional: sort boundaries
        total_delta = dp[n][k][0]
        total_boundary = dp[n][k][1]
        ratio = total_delta / total_boundary if total_boundary != 0 else 0
        results[k] = {
            "boundaries": boundaries,
            "cost": total_delta,
            "ratio": ratio
        }
    return results

# Example Usage
nums = [1, 3, 5, 7, 9]
max_k = 3
result = min_delta_boundaries_optimized(nums, max_k)
for k in sorted(result.keys()):
    print(f"k={k}: {result[k]}")

k=1: {'boundaries': [9], 'cost': 20, 'ratio': 0.4444444444444444}
k=2: {'boundaries': [], 'cost': inf, 'ratio': inf}
k=3: {'boundaries': [], 'cost': inf, 'ratio': inf}
